In [1]:
import re
import math
import os
import pandas as pd

# =============================================================================
# 1. Data Parser
# =============================================================================
def parse_dat_file(dat_filepath):
    # Defaults matching the latest logic
    params = {
        'scaleLength': 635.0,
        'stringSpacing': 10.8,
        'timePerMm': 0.25,
        's_max': 6, 'f_max': 24, 'p_max': 24,
        'fingerLengths_mm': [70.0, 75.0, 70.0, 50.0],
        'fingerDifficulty': [1.0, 1.5, 2.0, 3.0],
        'natural_min_spread_mm': [20.0, 40.0, 60.0],
        'availableTime': [],
        # Weights
        'd_PC': 1.5, 'd_SC': 1.0, 'd_ST': 1.0, 'd_DH': 1.0, 'd_NO': 10.0,
        'alpha_PC': 0.2, 'alpha_SC': 0.2, 'alpha_ST': 0.2, 'alpha_DH': 0.2, 'alpha_NO': 0.2
    }

    print(f"DEBUG: Parsing .dat file: {dat_filepath}")
    if not os.path.exists(dat_filepath):
        print("WARNING: File not found. Using defaults.")
        return params

    try:
        content = ""
        for enc in ['utf-8-sig', 'utf-8', 'cp932']:
            try:
                with open(dat_filepath, 'r', encoding=enc) as f:
                    content = f.read()
                break
            except UnicodeDecodeError: continue

        def get_val(name, cast_func):
            pat = re.compile(rf'{name}\s*=\s*([-+]?[0-9]*\.?[0-9]+);?')
            m = pat.search(content)
            if m: return cast_func(m.group(1))
            return None

        def get_arr(name):
            pat = re.compile(rf'{name}\s*=\s*\[(.*?)\];?', re.DOTALL)
            m = pat.search(content)
            if m:
                txt = re.sub(r'//.*', '', m.group(1)).replace('\n', ' ')
                return [float(v) for v in txt.split(',') if v.strip()]
            return None

        for k in params.keys():
            if isinstance(params[k], list):
                val = get_arr(k)
                if val: params[k] = val
            else:
                val = get_val(k, float)
                if val is not None: params[k] = val
        
        params['s_max'] = int(params['s_max'])
        params['f_max'] = int(params['f_max'])
        
        return params
    except Exception as e:
        print(f"ERROR: Parsing failed ({e}). Using defaults.")
        return params

def parse_solution_file(solution_filepath):
    notes_data = []
    if not os.path.exists(solution_filepath): return []
    
    try:
        lines = []
        for enc in ['utf-8-sig', 'utf-8', 'cp932']:
            try:
                with open(solution_filepath, 'r', encoding=enc) as f:
                    lines = f.readlines()
                break
            except UnicodeDecodeError: continue

        header_found = False
        for line in lines:
            line = line.strip()
            if not line: continue
            if not header_found:
                if "Note" in line and "String" in line: header_found = True
                continue
            if line.startswith('---'): break
            
            parts = re.split(r'\s+', line)
            if parts[0] == "Note": continue
            if len(parts) >= 5:
                try:
                    # Handle Rest Note 'R'
                    if parts[0] == 'R':
                         note = {
                            'note': 'R', 'string': 0, 'fret': 0, 'hand': 0, 'finger': 0
                        }
                    else:
                        note = {
                            'note': parts[0],
                            'string': int(parts[1]),
                            'fret': int(parts[2]),
                            'hand': int(parts[3]),
                            'finger': 0 if parts[4] == 'x' else int(parts[4])
                        }
                    notes_data.append(note)
                except ValueError: continue
        return notes_data
    except: return []

# =============================================================================
# 2. Updated Model Logic (Matching Modified Python)
# =============================================================================
class GuitarModelReproducer:
    def __init__(self, params):
        self.p = params
        
        self.fingerLengths = [0.0] + self.p['fingerLengths_mm']
        self.fingerDiff = self.p['fingerDifficulty']
        self.knuckleDist = [0.0, 0.0] + self.p['natural_min_spread_mm']
        
        self.limitRatios = [0.0, 1.0, 1.05, 1.05, 1.2]
        
        # Updated Base Factors: Index(0.9), Middle(0.9), Ring(0.8), Pinky(1.0)
        self.baseFactors = [0.0, 0.9, 0.9, 0.8, 1.0]
        
        self.stringTension = {1: 1.0, 2: 1.0, 3: 1.0, 4: 1.2, 5: 1.4, 6: 1.6}
        
        self.deg_to_rad = math.pi / 180.0
        self.rad_to_deg = 180.0 / math.pi
        self.breakAngleDeg = 15.0
        self.tanTheta = math.tan(self.breakAngleDeg * self.deg_to_rad)
        
        sl = self.p['scaleLength']
        self.fret_pos = [sl - (sl / (2 ** (f / 12.0))) for f in range(self.p['f_max'] + 1)]

    def calculate_costs(self, hand_pos, finger_idx, string, fret):
        """
        Returns (Stretch_Cost, Physics_Cost)
        """
        if fret < hand_pos: return (100000.0, 0.0)

        pivotX = self.fret_pos[hand_pos]
        pivotY = 0.0 

        # --- Updated Rotation Logic ---
        maxRotation = 40.0 # Updated from 45.0
        
        # Base rotation: Str 6 -> 0, Str 1 -> -40
        baseTheta = maxRotation * (string - 6.0) / 5.0
        
        # Fret-based rotation (Max 15 deg at fret 17, weighted by string)
        # Only String 6 changes by 15 deg, String 1 changes by 0 deg.
        # String indices are 1 to 6.
        fretRotationMax = (fret / 17.0) * 15.0
        stringFactor = (string - 1.0) / 5.0 # Str 1 -> 0.0, Str 6 -> 1.0
        fretRotation = fretRotationMax * stringFactor
        
        thetaDeg = baseTheta - fretRotation
        thetaRad = thetaDeg * self.deg_to_rad

        d = self.knuckleDist[finger_idx]
        kX = pivotX + (d * math.cos(thetaRad))
        kY = pivotY + (d * math.sin(thetaRad))

        tX = self.fret_pos[fret]
        tY = (string - 1) * self.p['stringSpacing']

        vX = tX - kX; vY = tY - kY
        dist = math.sqrt(vX*vX + vY*vY)

        # Angle Calculation
        dotProd = vX * -math.sin(thetaRad) + vY * math.cos(thetaRad)
        cosDelta = dotProd / dist if dist > 0.001 else 1.0
        cosDelta = max(-1.0, min(1.0, cosDelta))
        deltaDeg = math.acos(cosDelta) * self.rad_to_deg

        # Effective Length Correction
        neckFactor = 1.0
        if fret >= 15:
            ratioHigh = (fret - 15.0) / (self.p['f_max'] - 15.0)
            if ratioHigh > 1.0: ratioHigh = 1.0
            neckFactor = 1.0 - (0.15 * ratioHigh)
        
        eff_len = self.fingerLengths[finger_idx] * self.baseFactors[finger_idx] * neckFactor
        limit = self.limitRatios[finger_idx]

        if eff_len > 0 and dist > eff_len * limit:
            return (100000.0, 0.0)

        # --- Stretch Cost (Geometric) ---
        distCost = 0.0
        if eff_len > 0:
            ratio = dist / eff_len
            if ratio < 0.3: 
                distCost = (0.3 - ratio) * 5.0
            elif ratio <= 0.6: # Updated Threshold from 0.7 to 0.6
                distCost = 0.0
            else:
                distCost = (ratio - 0.6) * 10.0
                if ratio > 1.0: distCost += (ratio - 1.0) * 20.0

        # Updated Max Angle: Reduced by 10 deg
        baseMaxAngle = 20.0 if finger_idx == 4 else 15.0
        maxAngle = baseMaxAngle
        if fret >= 12:
            maxAngle = max(10.0, baseMaxAngle - (fret - 12) * 0.5)
        
        angleCost = (deltaDeg - maxAngle) * 0.5 if deltaDeg > maxAngle else 0.0

        # Updated ST Cost: Removed diff multiplier/addition
        st_val = distCost + angleCost

        # --- Physics Cost (DH) ---
        diff = self.fingerDiff[finger_idx - 1]
        
        a_mm = 0.5 + (2.0 * fret / 12.0)
        l_mm = self.p['scaleLength'] - self.fret_pos[fret]
        tau = self.stringTension.get(string, 1.0)
        
        if l_mm <= 0: forceIndex = 0
        else: forceIndex = (2.0 * tau * a_mm) / (l_mm * self.tanTheta)
        
        ph_val = (forceIndex * 200.0) * diff

        return (st_val, ph_val)

    def calculate_total(self, notes_data):
        total = {'PC': 0.0, 'SC': 0.0, 'ST': 0.0, 'DH': 0.0, 'NO': 0.0}
        details = []
        
        num_notes = len(notes_data)
        # Default available time if empty
        avail_time = self.p['availableTime'] if self.p['availableTime'] else [30.0] * (num_notes + 10)
        
        prev_idx_map = {}
        acc_time = {}

        # Time Calculation
        for i in range(num_notes):
            is_rest = (notes_data[i]['note'] == 'R')
            if not is_rest:
                p = i - 1
                t_sum = 0.0
                while p >= 0 and notes_data[p]['note'] == 'R':
                    t_sum += avail_time[p]
                    p -= 1
                if p >= 0:
                    prev_idx_map[i] = p
                    acc_time[i] = t_sum + avail_time[p]
                else:
                    prev_idx_map[i] = i
                    acc_time[i] = 10000.0
            else:
                acc_time[i] = 0.0
                prev_idx_map[i] = -1

        # Cost Calculation Loop
        for i in range(num_notes):
            curr = notes_data[i]
            row = {'PC': 0.0, 'SC': 0.0, 'ST': 0.0, 'DH': 0.0, 'NO': 0.0}
            
            # Initialize time_penalty for safe access even if Rest
            time_penalty = 0.0
            
            if curr['note'] != 'R':
                time_penalty = min(1.0, 50.0 / (acc_time[i] + 0.01))
                
                if curr['fret'] == 0: row['NO'] = 1.0

                if curr['finger'] in [1,2,3,4] and curr['fret'] > 0:
                    st_raw, ph_raw = self.calculate_costs(
                        curr['hand'], curr['finger'], curr['string'], curr['fret']
                    )
                    if st_raw >= 50000: st_raw = 100000.0
                    row['ST'] = st_raw
                    row['DH'] = ph_raw

                prev_i = prev_idx_map.get(i, i)
                if prev_i != i and prev_i != -1:
                    prev = notes_data[prev_i]
                    row['PC'] = abs(self.fret_pos[prev['hand']] - self.fret_pos[curr['hand']])
                    row['SC'] = self.p['stringSpacing'] * abs(prev['string'] - curr['string'])
                    
                    if (curr['finger'] == 1 and prev['finger'] == 1 and 
                        curr['fret'] == prev['fret'] and abs(curr['string'] - prev['string']) >= 1):
                        row['SC'] -= (self.p['stringSpacing'] * 0.9)

            # Weight Application
            w_pc = row['PC'] * time_penalty * self.p['d_PC'] * self.p['alpha_PC']
            w_sc = row['SC'] * time_penalty * self.p['d_SC'] * self.p['alpha_SC']
            w_st = row['ST'] * self.p['d_ST'] * self.p['alpha_ST'] 
            w_dh = row['DH'] * self.p['d_DH'] * self.p['alpha_DH']
            w_no = row['NO'] * self.p['d_NO'] * self.p['alpha_NO']

            total['PC'] += w_pc
            total['SC'] += w_sc
            total['ST'] += w_st
            total['DH'] += w_dh
            total['NO'] += w_no
            
            sum_weighted = w_pc + w_sc + w_st + w_dh + w_no
            details.append({
                'Index': i+1, 'Note': curr['note'],
                'PC_w': w_pc, 'SC_w': w_sc, 'ST_w': w_st, 'PH_w': w_dh,
                'ST_raw': row['ST'], 'PH_raw': row['DH'],
                'Total': sum_weighted
            })

        grand_total = sum(total.values())
        return total, grand_total, pd.DataFrame(details)

if __name__ == "__main__":
    # Manual Execution for Testing
    dat_file = "datas/complex_solo.dat"
    sol_file = "complex_opt.txt"
    
    # Create dummy files if not exist for demo purposes (optional)
    # Here we assume the user will upload or has uploaded the files.
    
    if os.path.exists(dat_file) and os.path.exists(sol_file):
        params = parse_dat_file(dat_file)
        model = GuitarModelReproducer(params)
        notes = parse_solution_file(sol_file)
        
        if notes:
            bd, total, df = model.calculate_total(notes)
            print("\n" + "="*60)
            print(" --- Updated Cost Breakdown (Logic Modified) ---")
            print(f"  Position Change (PC): {bd['PC']:.4f}")
            print(f"  String Change   (SC): {bd['SC']:.4f}")
            print(f"  Geo Stretch     (ST): {bd['ST']:.4f}")
            print(f"  Phys Action     (DH): {bd['DH']:.4f}")
            print(f"  Open String     (NO): {bd['NO']:.4f}")
            print("-" * 40)
            print(f"  Total Cost (Sum)    : {total:.4f}")
            print("="*60)
            
            if not df.empty:
                print("\n[Details - Top 20]")
                pd.set_option('display.max_columns', None)
                pd.set_option('display.width', 1000)
                print(df.head(20).to_string(index=False, float_format="%.3f"))
    else:
        print("Please upload 'datas/complex_solo.dat' and 'complex_opt.txt' to run the calculation.")

DEBUG: Parsing .dat file: datas/complex_solo.dat

 --- Updated Cost Breakdown (Logic Modified) ---
  Position Change (PC): 171.2491
  String Change   (SC): 64.1255
  Geo Stretch     (ST): 125.0347
  Phys Action     (DH): 125.6986
  Open String     (NO): 0.0000
----------------------------------------
  Total Cost (Sum)    : 486.1080

[Details - Top 20]
 Index Note   PC_w  SC_w  ST_w  PH_w  ST_raw  PH_raw  Total
     1   A4  0.000 0.000 1.708 0.545   7.115   9.076  2.252
     2   A4  0.000 0.000 1.708 0.545   7.115   9.076  2.252
     3   A4  0.000 0.000 1.708 0.545   7.115   9.076  2.252
     4   G4  0.000 1.728 0.597 0.987   2.487  16.456  3.312
     5   F4  0.000 0.000 0.231 0.545   0.962   9.076  0.775
     6   E4  0.000 0.000 1.461 0.474   6.088   7.907  1.936
     7   F4  0.000 0.000 0.736 0.653   3.068  10.891  1.390
     8   E4  0.000 0.000 1.461 0.474   6.088   7.907  1.936
     9   D4  6.357 1.728 1.212 1.185   5.050  19.748 10.482
    10  Cs4  0.000 0.000 0.571 0.895   2.380 